In [7]:
import snowflake.connector
import pandas as pd
import plotly.express as px

conn = snowflake.connector.connect(
    account='UFVFJXF-OLB23785',
    user='POLONICK13',
    private_key_file=r'C:\Users\Footsman\rsa_key.p8',
    role='ACCOUNTADMIN',
    warehouse='COMPUTE_WH',
    database='WALMART_DB',
    schema='ANALYTICS',
)
print("connected")

c:\Users\Footsman\AppData\Local\Programs\Python\Python313\Lib\site-packages\snowflake\connector\connection\_connection.py:135: FutureWarning: enable_server_session_keep_alive_auto_detection was not set and defaults to True. In a future version, the default will change to None. Please provide an explicit value: True = check for running async queries before logout (queries are preserved); False/None = always send logout on close (async queries may be terminated by server). Logout behavior can also be overridden with server_session_keep_alive. See the connection parameter docs for more info.
  options = self.config.to_proto_options(


connected


In [8]:
BASE = """
    from walmart_fact_table f
    join walmart_date_dim  d on f.date_id = d.date_id
    join walmart_store_dim s on f.store_id = s.store_id
                            and f.dept_id  = s.dept_id
    where f.is_current
"""

def q(sql):
    cur = conn.cursor()
    try:
        cur.execute(sql)
        df = cur.fetch_pandas_all()
    finally:
        cur.close()
    df.columns = [c.lower() for c in df.columns]
    return df

In [12]:
df = q(f"""
    select f.store_id, d.isholiday, sum(f.store_weekly_sales)::float as sales
    {BASE}
    group by 1, 2
    order by 1
""")
px.bar(df, x='store_id', y='sales', color='isholiday', barmode='group',
       title='Weekly sales by store and holiday')

In [20]:
df = q(f"""
    select year(d.store_date) as yr,
           round(f.temperature / 10) * 10 as temp_bucket,
           sum(f.store_weekly_sales)::float as sales
    {BASE}
    group by 1, 2
    order by 2
""")
df['yr'] = df['yr'].astype(str)
px.bar(df, x='temp_bucket', y='sales', color='yr', barmode='group',
       title='Weekly sales by temperature and year')

In [9]:
df = q(f"""
    select f.dept_id, sum(f.store_weekly_sales)::float as sales
    {BASE}
    group by f.dept_id
    order by f.dept_id
""")

print(f"total: {df['sales'].sum()/1e9:.2f}bn")
px.bar(df, x='dept_id', y='sales', title='Weekly sales by department')

total: 6.74bn


In [13]:
df = q(f"""
    select f.store_size, sum(f.store_weekly_sales)::float as sales
    {BASE}
    group by 1
    order by 1
""")
px.area(df, x='store_size', y='sales', title='Weekly sales by store size')

In [14]:
df = q(f"""
    select month(d.store_date) as mon, s.store_type,
           sum(f.store_weekly_sales)::float as sales
    {BASE}
    group by 1, 2
    order by 1
""")
fig = px.line(df, x='mon', y='sales', color='store_type', markers=True,
              title='Weekly sales by store type and month')
fig.update_xaxes(dtick=1)
fig

In [15]:
df = q(f"""
    select year(d.store_date) as yr, f.store_id,
           sum(f.markdown1)::float as m1, sum(f.markdown2)::float as m2,
           sum(f.markdown3)::float as m3, sum(f.markdown4)::float as m4,
           sum(f.markdown5)::float as m5
    {BASE}
    group by 1, 2
    order by 1, 2
""")

by_year = df.groupby('yr', as_index=False)[['m1','m2','m3','m4','m5']].sum()
long = by_year.melt(id_vars='yr', var_name='markdown', value_name='amount')
long['markdown'] = long['markdown'].str.replace('m', 'MarkDown', regex=False)
long['yr'] = long['yr'].astype(str)
px.bar(long, x='yr', y='amount', color='markdown', barmode='group',
       title='Markdown totals by year')

In [16]:
df = q(f"""
    select s.store_type, sum(f.store_weekly_sales)::float as sales
    {BASE}
    group by 1
    order by 1
""")
px.pie(df, names='store_type', values='sales', hole=0.4,
       title='Share of weekly sales by store type')

In [17]:
df = q(f"""
    select year(d.store_date) as yr, avg(f.fuel_price)::float as avg_fuel_price
    {BASE}
    group by 1
    order by 1
""")
df['yr'] = df['yr'].astype(str)
px.bar(df, x='yr', y='avg_fuel_price', text_auto='.3f',
       title='Average fuel price by year')

In [18]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

yr = q(f"select year(d.store_date)  as k, sum(f.store_weekly_sales)::float as sales {BASE} group by 1 order by 1")
mo = q(f"select month(d.store_date) as k, sum(f.store_weekly_sales)::float as sales {BASE} group by 1 order by 1")
dy = q(f"select day(d.store_date)   as k, sum(f.store_weekly_sales)::float as sales {BASE} group by 1 order by 1")

fig = make_subplots(rows=1, cols=3,
                    subplot_titles=('By year', 'By month', 'By day of month'))
fig.add_trace(go.Bar(x=yr['k'].astype(str), y=yr['sales'], showlegend=False), 1, 1)
fig.add_trace(go.Bar(x=mo['k'], y=mo['sales'], showlegend=False), 1, 2)
fig.add_trace(go.Bar(x=dy['k'], y=dy['sales'], showlegend=False), 1, 3)
fig.update_layout(title='Weekly sales by year, month and day of month')
fig

In [19]:
df = q(f"""
    select round(f.cpi, 1)::float as cpi, sum(f.store_weekly_sales)::float as sales
    {BASE}
    and f.cpi is not null
    group by 1
    order by 1
""")
px.line(df, x='cpi', y='sales', markers=True, title='Weekly sales by CPI')